In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
print("Libraries imported")
matches = pd.read_csv('../data/matches.csv')
deliveries = pd.read_csv('../data/deliveries.csv')
print(f"matches shape : {matches.shape}")
print(f"deliveries shape : {deliveries.shape}")
print("\n MATCHES COLUMNS ")
print(matches.columns.tolist())
print("\n DELIVERIES COLUMNS ")
print(deliveries.columns.tolist())
print("\n MATCHES NULL COUNT ")
print(matches.isnull().sum()[matches.isnull().sum() > 0])
print("\n DELIVERIES NULL COUNT ")
print(deliveries.isnull().sum()[deliveries.isnull().sum() > 0])
df_matches = matches.copy()
df_matches['date'] = pd.to_datetime(df_matches['date'])
team_name_map = {
    'Delhi Daredevils'       : 'Delhi Capitals',
    'Deccan Chargers'        : 'Sunrisers Hyderabad',
    'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Kings XI Punjab'        : 'Punjab Kings',
}
for col in ['team1', 'team2', 'toss_winner', 'winner']:
    df_matches[col] = df_matches[col].replace(team_name_map)
df_matches['result_margin'] = df_matches['result_margin'].fillna(0)
df_matches = df_matches.dropna(subset=['winner'])
df_matches['season'] = df_matches['date'].dt.year
print(f"Cleaned matches shape: {df_matches.shape}")
print("\nSeason distribution:")
print(df_matches['season'].value_counts().sort_index())
df_del = deliveries.copy()
for col in ['batting_team', 'bowling_team']:
    df_del[col] = df_del[col].replace(team_name_map)
df_del['player_dismissed'] = df_del['player_dismissed'].fillna('not_out')
df_del['dismissal_kind']   = df_del['dismissal_kind'].fillna('none')
df_del['fielder']          = df_del['fielder'].fillna('none')
if 'match_id' not in df_del.columns and 'id' in df_del.columns:
    df_del = df_del.rename(columns={'id': 'match_id'})
print(f"Cleaned deliveries shape: {df_del.shape}")
print("\nNull check after cleaning:")
print(df_del.isnull().sum()[df_del.isnull().sum() > 0])
match_season = df_matches[['id', 'season', 'date']].copy()
df_del = df_del.merge(match_season, left_on='match_id', right_on='id', how='left')
df_del = df_del.drop(columns=['id'])
print("Merged deliveries sample:")
print(df_del[['match_id', 'season', 'batting_team', 'batter', 'batsman_runs']].head())
df_matches.to_csv('../data/matches_clean.csv',    index=False)
df_del.to_csv('../data/deliveries_clean.csv', index=False)
print("\n Cleaned files saved to data/")
print(f"  matches_clean.csv    → {df_matches.shape}")
print(f"  deliveries_clean.csv → {df_del.shape}")

Libraries imported
matches shape : (1095, 20)
deliveries shape : (260920, 17)

 MATCHES COLUMNS 
['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']

 DELIVERIES COLUMNS 
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']

 MATCHES NULL COUNT 
city                 51
player_of_match       5
winner                5
result_margin        19
target_runs           3
target_overs          3
method             1074
dtype: int64

 DELIVERIES NULL COUNT 
extras_type         246795
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64
Cleaned matches shape: (1090, 20)

Season distribution:
season
200